In [1]:
# In a VS Code terminal, run once if needed:  %pip install -r requirements.txt
# PySpark also requires a compatible Java runtime (Java 17+ for current Spark).
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DoubleType, DateType)

spark = (SparkSession.builder
         .appName("RetailDataWarehouseRevision")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.version

'4.3.0'

In [2]:
data_path = Path("retail_orders.csv")
raw = (spark.read
       .option("header", "true")
       .csv(str(data_path)))
raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [3]:
raw.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999       |0.05        |Credit Card     |COMPLETED   |
|ORD00002|2025-01-23|1003       |Kabir Rao    |Bengaluru|Karnataka     |203       |USB-C Hub     |Accessories|3       |1899      |0.1         |Debit Card      |COMPLETED   |
|ORD00003|2025-02-03|1010       |Zara Joshi   |Jaipur   |Rajasthan     |201       |Wireless Mouse|Accessories|2       |799       |

In [4]:
raw.count()

300

In [5]:
len(raw.columns)

14

In [6]:
raw.show(2)

+--------+----------+-----------+-------------+---------+---------+----------+------------+-----------+--------+----------+------------+--------------+------------+
|order_id|order_date|customer_id|customer_name|     city|    state|product_id|product_name|   category|quantity|unit_price|discount_pct|payment_method|order_status|
+--------+----------+-----------+-------------+---------+---------+----------+------------+-----------+--------+----------+------------+--------------+------------+
|ORD00001|2025-01-12|       1008|    Myra Nair|    Kochi|   Kerala|       206|   Desk Lamp|  Furniture|       4|       999|        0.05|   Credit Card|   COMPLETED|
|ORD00002|2025-01-23|       1003|    Kabir Rao|Bengaluru|Karnataka|       203|   USB-C Hub|Accessories|       3|      1899|         0.1|    Debit Card|   COMPLETED|
+--------+----------+-----------+-------------+---------+---------+----------+------------+-----------+--------+----------+------------+--------------+------------+
only showi

In [7]:
raw.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status']

In [8]:
updated_schema = StructType([
    StructField("order_id",StringType(),True),
    StructField("order_date",StringType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("customer_name",StringType(),True),
    StructField("city",StringType(),True),
    StructField("state",StringType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("category",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DoubleType(),True),
    StructField("discount_pct",DoubleType(),True),
    StructField("payment_method",StringType(),True),
    StructField("order_status",StringType(),True)

])

raw = spark.read.csv(
    str(data_path),
    header = True,
    schema=updated_schema
)

raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [9]:
raw = raw.withColumn(
    "order_date",
    F.to_date("order_date","yyyy-MM-dd")
)

In [10]:
raw.select("order_date").show(3)

+----------+
|order_date|
+----------+
|2025-01-12|
|2025-01-23|
|2025-02-03|
+----------+
only showing top 3 rows


In [11]:
raw = raw.withColumn(
    "ingested_time",
    F.current_timestamp()
)

In [12]:
raw.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|ingested_time             |
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999.0     |0.05        |Credit Card     |COMPLETED   |2026-11-15 15:51:42.456917|
|ORD00002|2025-01-23|1003       |Kabir Rao    |Bengaluru|Karnataka     |203       |USB-C Hub     |Accessories|3       |1899.0    |0.1         |Debit Card      |COMPLETED   |2026-11-15 15:51:42.456

In [13]:
raw = raw.withColumn(
    "grass_amount", F.col("quantity") * F.col("unit_price")
)
raw.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|ingested_time             |grass_amount|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999.0     |0.05        |Credit Card     |COMPLETED   |2026-11-15 15:51:42.667549|3996.0      |
|ORD00002|2025-01-23|1003       |Kabir Rao    |Bengaluru|Karnataka     |203       |USB-C Hub     |Accessories|3       |1899.0    |0.1         |D

In [14]:
raw.select(["discount_pct", "payment_method", "grass_amount"]).show(5)

+------------+----------------+------------+
|discount_pct|  payment_method|grass_amount|
+------------+----------------+------------+
|        0.05|     Credit Card|      3996.0|
|         0.1|      Debit Card|      5697.0|
|        0.15|     Net Banking|      1598.0|
|         0.0|Cash on Delivery|       999.0|
|        0.05|             UPI|      7596.0|
+------------+----------------+------------+
only showing top 5 rows


In [15]:
raw = raw.withColumn(
    "discount_amount", F.col("grass_amount") * (1 - F.col("discount_pct"))
)

In [16]:
bronze = raw.withColumn(
    "ingested_time",
    F.current_timestamp()
)

In [17]:
bronze.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|ingested_time             |grass_amount|discount_amount|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999.0     |0.05        |Credit Card     |COMPLETED   |2026-11-15 15:51:42.972892|3996.0      |3796.2         |
|ORD00002|2025-01-23|1003       |Kabir Rao    |Bengaluru|Karnataka     |203     

In [18]:
with_amounts = bronze.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
)

In [19]:
with_amounts.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|ingested_time             |grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999.0     |0.05        |Credit Card     |COMPLETED   |2026-11-15 15:51:43.156417|3996.0      |3796.2         |3996.0      |
|ORD00002|2025-01-23|1003   

In [20]:
completed_order_df = with_amounts.filter(F.col("order_status") == "COMPLETED")

In [21]:
completed_order_df.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|ingested_time             |grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999.0     |0.05        |Credit Card     |COMPLETED   |2026-11-15 15:51:43.342106|3996.0      |3796.2         |3996.0      |
|ORD00002|2025-01-23|1003   

In [22]:
category_wise_sales = completed_order_df.groupBy("category").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum("quantity").alias("total_sales"),
    F.sum("gross_amount").alias("revenue"),
    F.avg("gross_amount").alias("avg_gross_amount")
)

In [23]:
category_wise_sales.show(5, truncate=False)

+-----------+------------+-----------+--------+-----------------+
|category   |total_orders|total_sales|revenue |avg_gross_amount |
+-----------+------------+-----------+--------+-----------------+
|Accessories|187         |466        |629734.0|3367.561497326203|
|Furniture  |94          |235        |234765.0|2497.5           |
+-----------+------------+-----------+--------+-----------------+



In [24]:
category_wise_sales = category_wise_sales.orderBy(
    F.desc("revenue")
)

In [25]:
category_wise_sales.show(5, truncate=False)

+-----------+------------+-----------+--------+-----------------+
|category   |total_orders|total_sales|revenue |avg_gross_amount |
+-----------+------------+-----------+--------+-----------------+
|Accessories|187         |466        |629734.0|3367.561497326203|
|Furniture  |94          |235        |234765.0|2497.5           |
+-----------+------------+-----------+--------+-----------------+



In [26]:
completed_order_df.show(5, truncate=False)

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city     |state         |product_id|product_name  |category   |quantity|unit_price|discount_pct|payment_method  |order_status|ingested_time             |grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------------+------------+---------------+------------+
|ORD00001|2025-01-12|1008       |Myra Nair    |Kochi    |Kerala        |206       |Desk Lamp     |Furniture  |4       |999.0     |0.05        |Credit Card     |COMPLETED   |2026-11-15 15:51:44.707805|3996.0      |3796.2         |3996.0      |
|ORD00002|2025-01-23|1003   

In [27]:
customer_totals = completed_order_df.groupBy(
    "customer_id",
    "customer_name",
    "city"
)

In [28]:
top_customers = customer_totals.agg(
    F.sum(F.col("gross_amount")).alias("total_revenue")
)

In [29]:
top_customers = top_customers.orderBy(F.desc("total_revenue"))

In [30]:
top_customers = top_customers.limit(5)

In [31]:
top_customers.show()

+-----------+-------------+---------+-------------+
|customer_id|customer_name|     city|total_revenue|
+-----------+-------------+---------+-------------+
|       1012|    Kiara Roy|   Indore|     182304.0|
|       1003|    Kabir Rao|Bengaluru|     131031.0|
|       1008|    Myra Nair|    Kochi|      91908.0|
|       1006|    Sara Khan|Hyderabad|      87354.0|
|       1004|   Anaya Iyer|  Chennai|      73508.0|
+-----------+-------------+---------+-------------+



In [32]:
completed_order_df.createOrReplaceTempView("completed_orders")
monthly = spark.sql("""
    SELECT date_format(order_date, 'yyyy-MM') AS month,
           COUNT(DISTINCT order_id) AS orders,
           ROUND(SUM(gross_amount), 2) AS revenue
    FROM completed_orders
    GROUP BY date_format(order_date, 'yyyy-MM')
    ORDER BY month
""")
monthly.show(12)

+-------+------+-------+
|  month|orders|revenue|
+-------+------+-------+
|2025-01|    26|83435.0|
|2025-02|    22|72544.0|
|2025-03|    25|75440.0|
|2025-04|    24|74540.0|
|2025-05|    23|72342.0|
|2025-06|    23|71141.0|
|2025-07|    24|72440.0|
|2025-08|    24|76138.0|
|2025-09|    21|61248.0|
|2025-10|    23|69345.0|
|2025-11|    22|59848.0|
|2025-12|    24|76038.0|
+-------+------+-------+



In [33]:
completed_order_df.createOrReplaceTempView("completed_orders_views")

In [34]:
data = spark.sql(
    """SELECT discount_pct, payment_method, grass_amount FROM completed_orders_views"""
)
data.show()

+------------+----------------+------------+
|discount_pct|  payment_method|grass_amount|
+------------+----------------+------------+
|        0.05|     Credit Card|      3996.0|
|         0.1|      Debit Card|      5697.0|
|        0.15|     Net Banking|      1598.0|
|         0.0|Cash on Delivery|       999.0|
|        0.05|             UPI|      7596.0|
|         0.1|     Credit Card|      2397.0|
|        0.15|      Debit Card|      1998.0|
|         0.0|     Net Banking|      1899.0|
|        0.05|Cash on Delivery|      3196.0|
|         0.1|             UPI|      2997.0|
|        0.15|     Credit Card|      3798.0|
|         0.0|      Debit Card|       799.0|
|        0.05|     Net Banking|      3996.0|
|         0.1|Cash on Delivery|      5697.0|
|        0.15|             UPI|      1598.0|
|         0.0|     Credit Card|       999.0|
|        0.05|      Debit Card|      7596.0|
|         0.1|     Net Banking|      2397.0|
|        0.15|Cash on Delivery|      1998.0|
|         

In [35]:
months = spark.sql("""
    select 
    date_format(order_date,'yyyy-MM') as Month,
    count(distinct(order_id)) as total_orders,
    sum(gross_amount) as revenue
    from completed_orders_views
    group by date_format(order_date,'yyyy-MM')
    order by month
""")

In [36]:
months.show()

+-------+------------+-------+
|  Month|total_orders|revenue|
+-------+------------+-------+
|2025-01|          26|83435.0|
|2025-02|          22|72544.0|
|2025-03|          25|75440.0|
|2025-04|          24|74540.0|
|2025-05|          23|72342.0|
|2025-06|          23|71141.0|
|2025-07|          24|72440.0|
|2025-08|          24|76138.0|
|2025-09|          21|61248.0|
|2025-10|          23|69345.0|
|2025-11|          22|59848.0|
|2025-12|          24|76038.0|
+-------+------------+-------+



In [37]:
completed_order_df.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount',
 'gross_amount']

In [38]:
product_sales = completed_order_df.groupBy("product_name","product_id","category")

In [39]:
product_sales = product_sales.agg(
    F.round(F.sum("gross_amount"), 2).alias("revenue")
)

In [40]:
product_sales.show()

+--------------+----------+-----------+--------+
|  product_name|product_id|   category| revenue|
+--------------+----------+-----------+--------+
|     USB-C Hub|       203|Accessories|444366.0|
|     Desk Lamp|       206|  Furniture|234765.0|
|Wireless Mouse|       201|Accessories|185368.0|
+--------------+----------+-----------+--------+



In [41]:
raw.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount']

In [42]:
window_space = Window.partitionBy("category").orderBy(F.desc("grass_amount"))

result = raw.withColumn(
    "window_dense_rank",F.dense_rank().over(window_space)
)



result.show()

+--------+----------+-----------+-------------+------+--------------+----------+------------+-----------+--------+----------+------------+----------------+------------+--------------------+------------+---------------+-----------------+
|order_id|order_date|customer_id|customer_name|  city|         state|product_id|product_name|   category|quantity|unit_price|discount_pct|  payment_method|order_status|       ingested_time|grass_amount|discount_amount|window_dense_rank|
+--------+----------+-----------+-------------+------+--------------+----------+------------+-----------+--------+----------+------------+----------------+------------+--------------------+------------+---------------+-----------------+
|ORD00005|2025-02-25|       1012|    Kiara Roy|Indore|Madhya Pradesh|       203|   USB-C Hub|Accessories|       4|    1899.0|        0.05|             UPI|   COMPLETED|2026-11-15 15:51:...|      7596.0|         7216.2|                1|
|ORD00017|2025-07-07|       1012|    Kiara Roy|Indor

In [43]:
window_space = Window.partitionBy("category").orderBy("unit_price")


result = raw.withColumn(
    "result_rank",
    F.rank().over(window_space)
)

result.show()

+--------+----------+-----------+-------------+-------+-----------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------+------------+---------------+-----------+
|order_id|order_date|customer_id|customer_name|   city|      state|product_id|  product_name|   category|quantity|unit_price|discount_pct|  payment_method|order_status|       ingested_time|grass_amount|discount_amount|result_rank|
+--------+----------+-----------+-------------+-------+-----------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------+------------+---------------+-----------+
|ORD00003|2025-02-03|       1010|   Zara Joshi| Jaipur|  Rajasthan|       201|Wireless Mouse|Accessories|       2|     799.0|        0.15|     Net Banking|   COMPLETED|2026-11-15 15:51:...|      1598.0|         1358.3|          1|
|ORD00006|2025-03-08|       1007|  Reyansh Das|Kolkata|West Bengal|       20

In [44]:
with_amounts.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount',
 'gross_amount']

In [45]:
dim_customer = with_amounts.select(
    "customer_id",
    "customer_name",
    "city",
    "state",

)

dim_customer.show()

+-----------+-------------+---------+--------------+
|customer_id|customer_name|     city|         state|
+-----------+-------------+---------+--------------+
|       1008|    Myra Nair|    Kochi|        Kerala|
|       1003|    Kabir Rao|Bengaluru|     Karnataka|
|       1010|   Zara Joshi|   Jaipur|     Rajasthan|
|       1005| Ishaan Gupta|    Delhi|         Delhi|
|       1012|    Kiara Roy|   Indore|Madhya Pradesh|
|       1007|  Reyansh Das|  Kolkata|   West Bengal|
|       1002|    Diya Shah|     Pune|   Maharashtra|
|       1009|  Arjun Patel|Ahmedabad|       Gujarat|
|       1004|   Anaya Iyer|  Chennai|    Tamil Nadu|
|       1011|   Vihaan Sen|  Lucknow| Uttar Pradesh|
|       1006|    Sara Khan|Hyderabad|     Telangana|
|       1001|  Aarav Mehta|   Mumbai|   Maharashtra|
|       1008|    Myra Nair|    Kochi|        Kerala|
|       1003|    Kabir Rao|Bengaluru|     Karnataka|
|       1010|   Zara Joshi|   Jaipur|     Rajasthan|
|       1005| Ishaan Gupta|    Delhi|         

In [46]:
x = dim_customer.groupBy("customer_id","customer_name","city","state").agg(
    F.count("*").alias("duplicated_customers")
).filter("duplicated_customers >1")
x.show()

+-----------+-------------+---------+--------------+--------------------+
|customer_id|customer_name|     city|         state|duplicated_customers|
+-----------+-------------+---------+--------------+--------------------+
|       1003|    Kabir Rao|Bengaluru|     Karnataka|                  25|
|       1012|    Kiara Roy|   Indore|Madhya Pradesh|                  25|
|       1002|    Diya Shah|     Pune|   Maharashtra|                  25|
|       1004|   Anaya Iyer|  Chennai|    Tamil Nadu|                  25|
|       1008|    Myra Nair|    Kochi|        Kerala|                  25|
|       1005| Ishaan Gupta|    Delhi|         Delhi|                  25|
|       1009|  Arjun Patel|Ahmedabad|       Gujarat|                  25|
|       1011|   Vihaan Sen|  Lucknow| Uttar Pradesh|                  25|
|       1006|    Sara Khan|Hyderabad|     Telangana|                  25|
|       1010|   Zara Joshi|   Jaipur|     Rajasthan|                  25|
|       1007|  Reyansh Das|  Kolkata| 

In [47]:
dim_customer = dim_customer.dropDuplicates(["customer_id"])

In [48]:
dim_customer.show()

+-----------+-------------+---------+--------------+
|customer_id|customer_name|     city|         state|
+-----------+-------------+---------+--------------+
|       1001|  Aarav Mehta|   Mumbai|   Maharashtra|
|       1002|    Diya Shah|     Pune|   Maharashtra|
|       1003|    Kabir Rao|Bengaluru|     Karnataka|
|       1004|   Anaya Iyer|  Chennai|    Tamil Nadu|
|       1005| Ishaan Gupta|    Delhi|         Delhi|
|       1006|    Sara Khan|Hyderabad|     Telangana|
|       1007|  Reyansh Das|  Kolkata|   West Bengal|
|       1008|    Myra Nair|    Kochi|        Kerala|
|       1009|  Arjun Patel|Ahmedabad|       Gujarat|
|       1010|   Zara Joshi|   Jaipur|     Rajasthan|
|       1011|   Vihaan Sen|  Lucknow| Uttar Pradesh|
|       1012|    Kiara Roy|   Indore|Madhya Pradesh|
+-----------+-------------+---------+--------------+



In [49]:
raw.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount']

In [50]:
dim_product = raw.select(
    "product_id",
    "product_name",
    "category",
    "quantity",
    
)

In [51]:
dim_product.show()

+----------+--------------+-----------+--------+
|product_id|  product_name|   category|quantity|
+----------+--------------+-----------+--------+
|       206|     Desk Lamp|  Furniture|       4|
|       203|     USB-C Hub|Accessories|       3|
|       201|Wireless Mouse|Accessories|       2|
|       206|     Desk Lamp|  Furniture|       1|
|       203|     USB-C Hub|Accessories|       4|
|       201|Wireless Mouse|Accessories|       3|
|       206|     Desk Lamp|  Furniture|       2|
|       203|     USB-C Hub|Accessories|       1|
|       201|Wireless Mouse|Accessories|       4|
|       206|     Desk Lamp|  Furniture|       3|
|       203|     USB-C Hub|Accessories|       2|
|       201|Wireless Mouse|Accessories|       1|
|       206|     Desk Lamp|  Furniture|       4|
|       203|     USB-C Hub|Accessories|       3|
|       201|Wireless Mouse|Accessories|       2|
|       206|     Desk Lamp|  Furniture|       1|
|       203|     USB-C Hub|Accessories|       4|
|       201|Wireless

In [52]:
duplicate_product = dim_product.groupBy("product_id","product_name","category","quantity").agg(
    F.count("*").alias("duplicated_products")
).filter("duplicated_products>1")

duplicate_product.show()

+----------+--------------+-----------+--------+-------------------+
|product_id|  product_name|   category|quantity|duplicated_products|
+----------+--------------+-----------+--------+-------------------+
|       201|Wireless Mouse|Accessories|       2|                 25|
|       201|Wireless Mouse|Accessories|       3|                 25|
|       201|Wireless Mouse|Accessories|       4|                 25|
|       203|     USB-C Hub|Accessories|       3|                 25|
|       203|     USB-C Hub|Accessories|       4|                 25|
|       206|     Desk Lamp|  Furniture|       2|                 25|
|       203|     USB-C Hub|Accessories|       1|                 25|
|       203|     USB-C Hub|Accessories|       2|                 25|
|       201|Wireless Mouse|Accessories|       1|                 25|
|       206|     Desk Lamp|  Furniture|       4|                 25|
|       206|     Desk Lamp|  Furniture|       1|                 25|
|       206|     Desk Lamp|  Furni

In [53]:
duplicate_product = dim_product.groupBy("product_id","product_name","category","quantity").agg(
    F.count("*").alias("duplicated_products")
).filter("duplicated_products>1")

duplicate_product.show()

+----------+--------------+-----------+--------+-------------------+
|product_id|  product_name|   category|quantity|duplicated_products|
+----------+--------------+-----------+--------+-------------------+
|       201|Wireless Mouse|Accessories|       2|                 25|
|       201|Wireless Mouse|Accessories|       3|                 25|
|       201|Wireless Mouse|Accessories|       4|                 25|
|       203|     USB-C Hub|Accessories|       3|                 25|
|       203|     USB-C Hub|Accessories|       4|                 25|
|       206|     Desk Lamp|  Furniture|       2|                 25|
|       203|     USB-C Hub|Accessories|       1|                 25|
|       203|     USB-C Hub|Accessories|       2|                 25|
|       201|Wireless Mouse|Accessories|       1|                 25|
|       206|     Desk Lamp|  Furniture|       4|                 25|
|       206|     Desk Lamp|  Furniture|       1|                 25|
|       206|     Desk Lamp|  Furni

In [54]:
duplicated_product =  duplicate_product.dropDuplicates(["product_id"])

In [55]:
dim_product = dim_product.dropDuplicates(["product_id"])
dim_product.count()

3

In [56]:
duplicated_product.show()

+----------+--------------+-----------+--------+-------------------+
|product_id|  product_name|   category|quantity|duplicated_products|
+----------+--------------+-----------+--------+-------------------+
|       201|Wireless Mouse|Accessories|       2|                 25|
|       203|     USB-C Hub|Accessories|       3|                 25|
|       206|     Desk Lamp|  Furniture|       2|                 25|
+----------+--------------+-----------+--------+-------------------+



In [57]:
completed_order_df.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount',
 'gross_amount']

In [58]:
dim_orders = completed_order_df.select(

    "order_id",
    "customer_id",
    "product_id",
    "unit_price",
    "discount_pct",
    "payment_method",
    "ingested_time",
    "gross_amount",
    "order_date",

)

In [59]:
dim_orders.show()

+--------+-----------+----------+----------+------------+----------------+--------------------+------------+----------+
|order_id|customer_id|product_id|unit_price|discount_pct|  payment_method|       ingested_time|gross_amount|order_date|
+--------+-----------+----------+----------+------------+----------------+--------------------+------------+----------+
|ORD00001|       1008|       206|     999.0|        0.05|     Credit Card|2026-11-15 15:51:...|      3996.0|2025-01-12|
|ORD00002|       1003|       203|    1899.0|         0.1|      Debit Card|2026-11-15 15:51:...|      5697.0|2025-01-23|
|ORD00003|       1010|       201|     799.0|        0.15|     Net Banking|2026-11-15 15:51:...|      1598.0|2025-02-03|
|ORD00004|       1005|       206|     999.0|         0.0|Cash on Delivery|2026-11-15 15:51:...|       999.0|2025-02-14|
|ORD00005|       1012|       203|    1899.0|        0.05|             UPI|2026-11-15 15:51:...|      7596.0|2025-02-25|
|ORD00006|       1007|       201|     79

In [60]:
duplicated_orders = dim_orders.groupBy(
    "order_id","customer_id","product_id","unit_price","discount_pct","payment_method","ingested_time","gross_amount","order_date").agg(
    F.count("*").alias("duplicated_orders")).filter("duplicated_orders>1")


In [61]:
duplicated_orders.show()

+--------+-----------+----------+----------+------------+--------------+-------------+------------+----------+-----------------+
|order_id|customer_id|product_id|unit_price|discount_pct|payment_method|ingested_time|gross_amount|order_date|duplicated_orders|
+--------+-----------+----------+----------+------------+--------------+-------------+------------+----------+-----------------+
+--------+-----------+----------+----------+------------+--------------+-------------+------------+----------+-----------------+



In [62]:
order_with_customer = dim_orders.join(
    dim_customer,
    on="customer_id",
    how="left"
)

In [63]:
order_with_customer.show()

+-----------+--------+----------+----------+------------+----------------+--------------------+------------+----------+-------------+---------+--------------+
|customer_id|order_id|product_id|unit_price|discount_pct|  payment_method|       ingested_time|gross_amount|order_date|customer_name|     city|         state|
+-----------+--------+----------+----------+------------+----------------+--------------------+------------+----------+-------------+---------+--------------+
|       1008|ORD00001|       206|     999.0|        0.05|     Credit Card|2026-11-15 15:51:...|      3996.0|2025-01-12|    Myra Nair|    Kochi|        Kerala|
|       1003|ORD00002|       203|    1899.0|         0.1|      Debit Card|2026-11-15 15:51:...|      5697.0|2025-01-23|    Kabir Rao|Bengaluru|     Karnataka|
|       1010|ORD00003|       201|     799.0|        0.15|     Net Banking|2026-11-15 15:51:...|      1598.0|2025-02-03|   Zara Joshi|   Jaipur|     Rajasthan|
|       1005|ORD00004|       206|     999.0|  

In [64]:
joined_df = order_with_customer.join(
    dim_product,
    on="product_id",
    how="left"
)

In [65]:
joined_df.show(truncate=False)

+----------+-----------+--------+----------+------------+----------------+--------------------------+------------+----------+-------------+---------+--------------+--------------+-----------+--------+
|product_id|customer_id|order_id|unit_price|discount_pct|payment_method  |ingested_time             |gross_amount|order_date|customer_name|city     |state         |product_name  |category   |quantity|
+----------+-----------+--------+----------+------------+----------------+--------------------------+------------+----------+-------------+---------+--------------+--------------+-----------+--------+
|206       |1008       |ORD00001|999.0     |0.05        |Credit Card     |2026-11-15 15:51:50.140366|3996.0      |2025-01-12|Myra Nair    |Kochi    |Kerala        |Desk Lamp     |Furniture  |4       |
|203       |1003       |ORD00002|1899.0    |0.1         |Debit Card      |2026-11-15 15:51:50.140366|5697.0      |2025-01-23|Kabir Rao    |Bengaluru|Karnataka     |USB-C Hub     |Accessories|3    

In [66]:
joined_df.columns

['product_id',
 'customer_id',
 'order_id',
 'unit_price',
 'discount_pct',
 'payment_method',
 'ingested_time',
 'gross_amount',
 'order_date',
 'customer_name',
 'city',
 'state',
 'product_name',
 'category',
 'quantity']

In [67]:
joined_df.select(
    "order_id",
    "customer_name",
    "product_name",
    "gross_amount",
).show(30)

+--------+-------------+--------------+------------+
|order_id|customer_name|  product_name|gross_amount|
+--------+-------------+--------------+------------+
|ORD00001|    Myra Nair|     Desk Lamp|      3996.0|
|ORD00002|    Kabir Rao|     USB-C Hub|      5697.0|
|ORD00003|   Zara Joshi|Wireless Mouse|      1598.0|
|ORD00004| Ishaan Gupta|     Desk Lamp|       999.0|
|ORD00005|    Kiara Roy|     USB-C Hub|      7596.0|
|ORD00006|  Reyansh Das|Wireless Mouse|      2397.0|
|ORD00007|    Diya Shah|     Desk Lamp|      1998.0|
|ORD00008|  Arjun Patel|     USB-C Hub|      1899.0|
|ORD00009|   Anaya Iyer|Wireless Mouse|      3196.0|
|ORD00010|   Vihaan Sen|     Desk Lamp|      2997.0|
|ORD00011|    Sara Khan|     USB-C Hub|      3798.0|
|ORD00012|  Aarav Mehta|Wireless Mouse|       799.0|
|ORD00013|    Myra Nair|     Desk Lamp|      3996.0|
|ORD00014|    Kabir Rao|     USB-C Hub|      5697.0|
|ORD00015|   Zara Joshi|Wireless Mouse|      1598.0|
|ORD00016| Ishaan Gupta|     Desk Lamp|       

In [68]:
from pyspark.sql import functions as F

# Check each table's join key for duplicates
dim_orders.groupBy("order_id").count().filter("count > 1").show()
dim_customer.groupBy("customer_id").count().filter("count > 1").show()
dim_product.groupBy("product_id").count().filter("count > 1").show()


+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [69]:
# Count missing values in each column
null_counts = with_amounts.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in with_amounts.columns
])

null_counts.show()


+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city|state|product_id|product_name|category|quantity|unit_price|discount_pct|payment_method|order_status|ingested_time|grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|       0|         0|          0|            0|   0|    0|         0|           0|       0|       0|         0|           0|             0|           0|            0|           0|              0|           0|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+------------

In [70]:
null_values  = raw.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in raw.columns
])

null_values.show()

+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+
|order_id|order_date|customer_id|customer_name|city|state|product_id|product_name|category|quantity|unit_price|discount_pct|payment_method|order_status|ingested_time|grass_amount|discount_amount|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+
|       0|         0|          0|            0|   0|    0|         0|           0|       0|       0|         0|           0|             0|           0|            0|           0|              0|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+



In [71]:
null_values_dim_customer = dim_customer.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in dim_customer.columns
])
null_values_dim_customer.show()

+-----------+-------------+----+-----+
|customer_id|customer_name|city|state|
+-----------+-------------+----+-----+
|          0|            0|   0|    0|
+-----------+-------------+----+-----+



In [73]:
duplicated = (
    with_amounts
    .groupBy('order_id')
    .count()
    .filter("count > 1")
)
duplicated.count()

0

In [74]:
with_amounts.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount',
 'gross_amount']

In [76]:
invalid_with_amounts = with_amounts.filter(
    (F.col("quantity") <= 0 ) |
    (F.col("unit_price")<0) |
    (F.col("discount_pct")< 0) |
    (F.col("gross_amount")< 0 )|
    (F.col("order_date").isNull())
)

invalid_with_amounts.show()

+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city|state|product_id|product_name|category|quantity|unit_price|discount_pct|payment_method|order_status|ingested_time|grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+



In [83]:
with_amounts = with_amounts.withColumn(
    "discount_amount",
    F.round(F.col("discount_amount"),2)
)

In [84]:
with_amounts.show()

+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|     city|         state|product_id|  product_name|   category|quantity|unit_price|discount_pct|  payment_method|order_status|       ingested_time|grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+---------+--------------+----------+--------------+-----------+--------+----------+------------+----------------+------------+--------------------+------------+---------------+------------+
|ORD00001|2025-01-12|       1008|    Myra Nair|    Kochi|        Kerala|       206|     Desk Lamp|  Furniture|       4|     999.0|        0.05|     Credit Card|   COMPLETED|2026-11-15 16:07:...|      3996.0|         3796.2|      3996.0|
|ORD00002|2025-01-23|       1003|    Kabir Rao|Benga

In [87]:
silver = with_amounts.filter(
    (F.col("quantity") > 0) & 
    (F.col("unit_price") >= 0) &
    (F.col("discount_pct").between(0, 1)) &
    (F.col("order_date").isNotNull())
)

silver.count()

300

In [89]:
null_in_silver = silver.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in silver.columns
])

null_in_silver.show()

+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city|state|product_id|product_name|category|quantity|unit_price|discount_pct|payment_method|order_status|ingested_time|grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|       0|         0|          0|            0|   0|    0|         0|           0|       0|       0|         0|           0|             0|           0|            0|           0|              0|           0|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+------------

In [90]:
silver.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount',
 'gross_amount']

In [94]:
duplicate_in_silver = (
    silver
    .groupBy("order_id","customer_id","product_id","unit_price","discount_pct","payment_method","ingested_time","gross_amount","order_date")
    .count()
    .filter("count > 1")
)

duplicate_in_silver.show()

+--------+-----------+----------+----------+------------+--------------+-------------+------------+----------+-----+
|order_id|customer_id|product_id|unit_price|discount_pct|payment_method|ingested_time|gross_amount|order_date|count|
+--------+-----------+----------+----------+------------+--------------+-------------+------------+----------+-----+
+--------+-----------+----------+----------+------------+--------------+-------------+------------+----------+-----+



In [95]:
silver = silver.dropDuplicates(["order_id"])

In [96]:
silver.count()

300

In [97]:
null_in_with_amounts = with_amounts.select(
[    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in with_amounts.columns]
)

null_in_with_amounts.show()

+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|order_id|order_date|customer_id|customer_name|city|state|product_id|product_name|category|quantity|unit_price|discount_pct|payment_method|order_status|ingested_time|grass_amount|discount_amount|gross_amount|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+-------------+------------+---------------+------------+
|       0|         0|          0|            0|   0|    0|         0|           0|       0|       0|         0|           0|             0|           0|            0|           0|              0|           0|
+--------+----------+-----------+-------------+----+-----+----------+------------+--------+--------+----------+------------+--------------+------------+------------

In [98]:
duplicated_order_id_in_with_amounts = (
    with_amounts
    .groupBy("order_id")
    .count()
    .filter("count > 1")
)

duplicated_order_id_in_with_amounts.show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [102]:
silver.select("order_date").distinct().count()

300

In [103]:
dim_date = silver.select("order_date").distinct()

In [122]:
dim_date = dim_date.withColumn(
    "date_key",
    F.date_format("order_date","yyyyMMdd").cast("int")
)

dim_date.show()

+----------+--------+----+-------+-----+----------+-----------+
|order_date|date_key|year|quarter|month|month_name|day_of_week|
+----------+--------+----+-------+-----+----------+-----------+
|2025-03-19|20250319|2025|      1|    3|     March|  Wednesday|
|2025-11-05|20251105|2025|      4|   11|  November|  Wednesday|
|2025-11-27|20251127|2025|      4|   11|  November|   Thursday|
|2025-02-23|20250223|2025|      1|    2|  February|     Sunday|
|2025-04-06|20250406|2025|      2|    4|     April|     Sunday|
|2025-04-17|20250417|2025|      2|    4|     April|   Thursday|
|2025-08-14|20250814|2025|      3|    8|    August|   Thursday|
|2025-11-08|20251108|2025|      4|   11|  November|   Saturday|
|2025-02-26|20250226|2025|      1|    2|  February|  Wednesday|
|2025-03-31|20250331|2025|      1|    3|     March|     Monday|
|2025-06-05|20250605|2025|      2|    6|      June|   Thursday|
|2025-04-20|20250420|2025|      2|    4|     April|     Sunday|
|2025-09-10|20250910|2025|      3|    9|

In [123]:
dim_date = dim_date.withColumns(
{    "year" : F.year("order_date"),
    "quarter" : F.quarter("order_date"),
    "month" : F.month("order_date"),
    "month_name" : F.date_format("order_date","MMMM"),
    "day_of_week": F.date_format("order_date","EEEE")
    }
)

dim_date.show()

+----------+--------+----+-------+-----+----------+-----------+
|order_date|date_key|year|quarter|month|month_name|day_of_week|
+----------+--------+----+-------+-----+----------+-----------+
|2025-03-19|20250319|2025|      1|    3|     March|  Wednesday|
|2025-11-05|20251105|2025|      4|   11|  November|  Wednesday|
|2025-11-27|20251127|2025|      4|   11|  November|   Thursday|
|2025-02-23|20250223|2025|      1|    2|  February|     Sunday|
|2025-04-06|20250406|2025|      2|    4|     April|     Sunday|
|2025-04-17|20250417|2025|      2|    4|     April|   Thursday|
|2025-08-14|20250814|2025|      3|    8|    August|   Thursday|
|2025-11-08|20251108|2025|      4|   11|  November|   Saturday|
|2025-02-26|20250226|2025|      1|    2|  February|  Wednesday|
|2025-03-31|20250331|2025|      1|    3|     March|     Monday|
|2025-06-05|20250605|2025|      2|    6|      June|   Thursday|
|2025-04-20|20250420|2025|      2|    4|     April|     Sunday|
|2025-09-10|20250910|2025|      3|    9|

In [124]:
dim_customer_gold = silver.select(
    "customer_id","customer_name","city","state"
).dropDuplicates(["customer_id"])

In [125]:
dim_customer_gold.show()

+-----------+-------------+---------+--------------+
|customer_id|customer_name|     city|         state|
+-----------+-------------+---------+--------------+
|       1001|  Aarav Mehta|   Mumbai|   Maharashtra|
|       1002|    Diya Shah|     Pune|   Maharashtra|
|       1003|    Kabir Rao|Bengaluru|     Karnataka|
|       1004|   Anaya Iyer|  Chennai|    Tamil Nadu|
|       1005| Ishaan Gupta|    Delhi|         Delhi|
|       1006|    Sara Khan|Hyderabad|     Telangana|
|       1007|  Reyansh Das|  Kolkata|   West Bengal|
|       1008|    Myra Nair|    Kochi|        Kerala|
|       1009|  Arjun Patel|Ahmedabad|       Gujarat|
|       1010|   Zara Joshi|   Jaipur|     Rajasthan|
|       1011|   Vihaan Sen|  Lucknow| Uttar Pradesh|
|       1012|    Kiara Roy|   Indore|Madhya Pradesh|
+-----------+-------------+---------+--------------+



In [126]:
dim_product_gold = silver.select(
    "product_id","product_name","category"
).dropDuplicates(['product_id'])
dim_product_gold.show()

+----------+--------------+-----------+
|product_id|  product_name|   category|
+----------+--------------+-----------+
|       201|Wireless Mouse|Accessories|
|       203|     USB-C Hub|Accessories|
|       206|     Desk Lamp|  Furniture|
+----------+--------------+-----------+



In [127]:
silver.columns

['order_id',
 'order_date',
 'customer_id',
 'customer_name',
 'city',
 'state',
 'product_id',
 'product_name',
 'category',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'order_status',
 'ingested_time',
 'grass_amount',
 'discount_amount',
 'gross_amount']

In [128]:
fact_sales_table = silver.select(
    "order_id",
    "order_date",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount_pct",
    "gross_amount",
    "order_status",
    "payment_method"

)

fact_sales_table = fact_sales_table.withColumn(
    "date_key",
    F.date_format("order_date","yyyyMMdd").cast("int")
)

fact_sales_table.show()

+--------+----------+-----------+----------+--------+----------+------------+------------+------------+----------------+--------+
|order_id|order_date|customer_id|product_id|quantity|unit_price|discount_pct|gross_amount|order_status|  payment_method|date_key|
+--------+----------+-----------+----------+--------+----------+------------+------------+------------+----------------+--------+
|ORD00001|2025-01-12|       1008|       206|       4|     999.0|        0.05|      3996.0|   COMPLETED|     Credit Card|20250112|
|ORD00002|2025-01-23|       1003|       203|       3|    1899.0|         0.1|      5697.0|   COMPLETED|      Debit Card|20250123|
|ORD00003|2025-02-03|       1010|       201|       2|     799.0|        0.15|      1598.0|   COMPLETED|     Net Banking|20250203|
|ORD00004|2025-02-14|       1005|       206|       1|     999.0|         0.0|       999.0|   COMPLETED|Cash on Delivery|20250214|
|ORD00005|2025-02-25|       1012|       203|       4|    1899.0|        0.05|      7596.0|

In [129]:
print("Date rows:", dim_date.count())
print("Customer rows:", dim_customer_gold.count())
print("Product rows:", dim_product_gold.count())
print("Sales rows:", fact_sales_table.count())

Date rows: 300
Customer rows: 12
Product rows: 3
Sales rows: 300


In [ ]:
completed_sales = fact_sales_table.filter(
    F.col("order_status") == "COMPLETED"
)
sales_with_year = completed_sales.join(
    dim_date.select("date_key","year"),
    on="date_key"
)

sales_with_year.show()

+--------+--------+----------+-----------+----------+--------+----------+------------+------------+------------+----------------+----+
|date_key|order_id|order_date|customer_id|product_id|quantity|unit_price|discount_pct|gross_amount|order_status|  payment_method|year|
+--------+--------+----------+-----------+----------+--------+----------+------------+------------+------------+----------------+----+
|20250112|ORD00001|2025-01-12|       1008|       206|       4|     999.0|        0.05|      3996.0|   COMPLETED|     Credit Card|2025|
|20250123|ORD00002|2025-01-23|       1003|       203|       3|    1899.0|         0.1|      5697.0|   COMPLETED|      Debit Card|2025|
|20250203|ORD00003|2025-02-03|       1010|       201|       2|     799.0|        0.15|      1598.0|   COMPLETED|     Net Banking|2025|
|20250214|ORD00004|2025-02-14|       1005|       206|       1|     999.0|         0.0|       999.0|   COMPLETED|Cash on Delivery|2025|
|20250225|ORD00005|2025-02-25|       1012|       203|  

In [133]:
sales_with_state = sales_with_year.join(
    dim_customer_gold.select("customer_id","state"),
    on="customer_id"
)

sales_with_state.show()

+-----------+--------+--------+----------+----------+--------+----------+------------+------------+------------+----------------+----+--------------+
|customer_id|date_key|order_id|order_date|product_id|quantity|unit_price|discount_pct|gross_amount|order_status|  payment_method|year|         state|
+-----------+--------+--------+----------+----------+--------+----------+------------+------------+------------+----------------+----+--------------+
|       1008|20250112|ORD00001|2025-01-12|       206|       4|     999.0|        0.05|      3996.0|   COMPLETED|     Credit Card|2025|        Kerala|
|       1003|20250123|ORD00002|2025-01-23|       203|       3|    1899.0|         0.1|      5697.0|   COMPLETED|      Debit Card|2025|     Karnataka|
|       1010|20250203|ORD00003|2025-02-03|       201|       2|     799.0|        0.15|      1598.0|   COMPLETED|     Net Banking|2025|     Rajasthan|
|       1005|20250214|ORD00004|2025-02-14|       206|       1|     999.0|         0.0|       999.0| 

In [134]:
sales_with_details = sales_with_state.join(
    dim_product_gold.select("product_id","category"),
    on="product_id"
)

sales_with_details.show()

+----------+-----------+--------+--------+----------+--------+----------+------------+------------+------------+----------------+----+--------------+-----------+
|product_id|customer_id|date_key|order_id|order_date|quantity|unit_price|discount_pct|gross_amount|order_status|  payment_method|year|         state|   category|
+----------+-----------+--------+--------+----------+--------+----------+------------+------------+------------+----------------+----+--------------+-----------+
|       206|       1008|20250112|ORD00001|2025-01-12|       4|     999.0|        0.05|      3996.0|   COMPLETED|     Credit Card|2025|        Kerala|  Furniture|
|       203|       1003|20250123|ORD00002|2025-01-23|       3|    1899.0|         0.1|      5697.0|   COMPLETED|      Debit Card|2025|     Karnataka|Accessories|
|       201|       1010|20250203|ORD00003|2025-02-03|       2|     799.0|        0.15|      1598.0|   COMPLETED|     Net Banking|2025|     Rajasthan|Accessories|
|       206|       1005|2025

In [136]:
report = sales_with_details.groupBy("year","category","state").agg(
    F.round(F.sum("gross_amount"),2).alias("revenue"),
    F.sum("quantity").alias("units")
)
report.show()

+----+-----------+--------------+--------+-----+
|year|   category|         state| revenue|units|
+----+-----------+--------------+--------+-----+
|2025|Accessories|     Karnataka|131031.0|   69|
|2025|  Furniture|         Delhi| 22977.0|   23|
|2025|Accessories|       Gujarat| 43677.0|   23|
|2025|  Furniture| Uttar Pradesh| 71928.0|   72|
|2025|Accessories|   Maharashtra| 19975.0|   25|
|2025|  Furniture|   Maharashtra| 47952.0|   48|
|2025|  Furniture|        Kerala| 91908.0|   92|
|2025|Accessories|     Rajasthan| 36754.0|   46|
|2025|Accessories|Madhya Pradesh|182304.0|   96|
|2025|Accessories|   West Bengal| 55131.0|   69|
|2025|Accessories|     Telangana| 87354.0|   46|
|2025|Accessories|    Tamil Nadu| 73508.0|   92|
+----+-----------+--------------+--------+-----+



In [137]:
report = report.orderBy(
    "year",
    "category",
    F.desc("revenue")
)

report.show()

+----+-----------+--------------+--------+-----+
|year|   category|         state| revenue|units|
+----+-----------+--------------+--------+-----+
|2025|Accessories|Madhya Pradesh|182304.0|   96|
|2025|Accessories|     Karnataka|131031.0|   69|
|2025|Accessories|     Telangana| 87354.0|   46|
|2025|Accessories|    Tamil Nadu| 73508.0|   92|
|2025|Accessories|   West Bengal| 55131.0|   69|
|2025|Accessories|       Gujarat| 43677.0|   23|
|2025|Accessories|     Rajasthan| 36754.0|   46|
|2025|Accessories|   Maharashtra| 19975.0|   25|
|2025|  Furniture|        Kerala| 91908.0|   92|
|2025|  Furniture| Uttar Pradesh| 71928.0|   72|
|2025|  Furniture|   Maharashtra| 47952.0|   48|
|2025|  Furniture|         Delhi| 22977.0|   23|
+----+-----------+--------------+--------+-----+



In [138]:
print({"source": raw.count(), "silver": silver.count(), "fact": fact_sales_table.count()})
report.explain(True)
# Optional: persist a reused DataFrame, then release it when done.
# from pyspark import StorageLevel
# silver.persist(StorageLevel.MEMORY_AND_DISK)
# silver.count()  # materialize cache
# silver.unpersist()

{'source': 300, 'silver': 300, 'fact': 300}
== Parsed Logical Plan ==
'Sort ['year ASC NULLS FIRST, 'category ASC NULLS FIRST, 'revenue DESC NULLS LAST], true
+- Aggregate [year#4734, category#6325, state#6009], [year#4734, category#6325, state#6009, round(sum(gross_amount#392), 2) AS revenue#6734, sum(quantity#173) AS units#6735L]
   +- Project [product_id#170, customer_id#166, date_key#5064, order_id#164, order_date#179, quantity#173, unit_price#174, discount_pct#175, gross_amount#392, order_status#177, payment_method#176, year#4734, state#6009, category#6325]
      +- Join Inner, (product_id#170 = product_id#6323)
         :- Project [customer_id#166, date_key#5064, order_id#164, order_date#179, product_id#170, quantity#173, unit_price#174, discount_pct#175, gross_amount#392, order_status#177, payment_method#176, year#4734, state#6009]
         :  +- Join Inner, (customer_id#166 = customer_id#6006)
         :     :- Project [date_key#5064, order_id#164, order_date#179, customer_id#1